[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/map-reduce.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239947-lesson-3-map-reduce)

# Map-Reduce

## 回顾

我们正在构建一个多 Agent 研究助手，将本课程的所有模块内容串联起来。

为了构建这个多 Agent 助手，我们一直在介绍几个 LangGraph 可控性主题。

我们刚刚介绍了并行化和子图。

## 目标

现在，我们将介绍 [map reduce](https://docs.langchain.com/oss/python/langgraph/use-graph-api#map-reduce-and-the-send-api)。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_deepseek langgraph

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 问题

Map-Reduce 操作对于高效的任务分解和并行处理至关重要。

它有两个阶段：

(1) `Map`（映射）—— 将任务分解为更小的子任务，并行处理每个子任务。

(2) `Reduce`（归约）—— 汇总所有已完成的并行子任务的结果。

让我们设计一个能做两件事的系统：

(1) `Map`—— 创建一组关于某个主题的笑话。

(2) `Reduce`—— 从列表中选出最佳笑话。

我们将使用 LLM 来完成笑话的生成和选择。

In [ ]:
from langchain_deepseek import ChatDeepSeek

# 我们将使用的提示
subjects_prompt = """Generate a list of 3 sub-topics that are all related to this overall topic: {topic}."""
joke_prompt = """Generate a joke about {subject}"""
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return the ID of the best one, starting 0 as the ID for the first joke. Jokes: \n\n  {jokes}"""

# LLM
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

## 状态

### 并行化笑话生成

首先，让我们定义图的入口点，它将：

* 接受用户输入的主题
* 从中生成一个笑话主题列表
* 将每个笑话主题发送到我们上面的笑话生成节点

我们的状态有一个 `jokes` 键，它将累积来自并行笑话生成的笑话。

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel

class Subjects(BaseModel):
    subjects: list[str]

class BestJoke(BaseModel):
    id: int
    
class OverallState(TypedDict):
    topic: str
    subjects: list
    jokes: Annotated[list, operator.add]
    best_selected_joke: str

为笑话生成子主题。

In [ ]:
def generate_topics(state: OverallState):
    prompt = subjects_prompt.format(topic=state["topic"])
    response = model.with_structured_output(Subjects).invoke(prompt)
    return {"subjects": response.subjects}

这就是魔法所在：我们使用 [Send](https://docs.langchain.com/oss/python/langgraph/graph-api/#send) 为每个主题创建一个笑话。

这非常有用！它可以自动并行化为任意数量的主题生成笑话。

* `generate_joke`：图中节点的名称
* `{"subject": s`}：要发送的状态

`Send` 允许你将任何状态传递给你想要的 `generate_joke`！它不必与 `OverallState` 对齐。

在这种情况下，`generate_joke` 使用自己的内部状态，我们可以通过 `Send` 来填充它。

In [ ]:
from langgraph.types import Send
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

### 笑话生成（Map）

现在，我们只需定义一个创建笑话的节点 `generate_joke`！

我们将它们写回到 `OverallState` 的 `jokes` 中！

这个键有一个 reducer，将组合列表。

In [ ]:
class JokeState(TypedDict):
    subject: str

class Joke(BaseModel):
    joke: str

def generate_joke(state: JokeState):
    prompt = joke_prompt.format(subject=state["subject"])
    response = model.with_structured_output(Joke).invoke(prompt)
    return {"jokes": [response.joke]}

### 最佳笑话选择（Reduce）

现在，我们添加挑选最佳笑话的逻辑。

In [ ]:
def best_joke(state: OverallState):
    jokes = "\n\n".join(state["jokes"])
    prompt = best_joke_prompt.format(topic=state["topic"], jokes=jokes)
    response = model.with_structured_output(BestJoke).invoke(prompt)
    return {"best_selected_joke": state["jokes"][response.id]}

## Compile

In [ ]:
from IPython.display import Image
from langgraph.graph import END, StateGraph, START

# 构建图：我们将所有内容组合在一起来构建我们的图
graph = StateGraph(OverallState)
graph.add_node("generate_topics", generate_topics)
graph.add_node("generate_joke", generate_joke)
graph.add_node("best_joke", best_joke)
graph.add_edge(START, "generate_topics")
graph.add_conditional_edges("generate_topics", continue_to_jokes, ["generate_joke"])
graph.add_edge("generate_joke", "best_joke")
graph.add_edge("best_joke", END)

# 编译图
app = graph.compile()
Image(app.get_graph().draw_mermaid_png())

In [ ]:
# 调用图：这里我们调用它以生成一个笑话列表
for s in app.stream({"topic": "animals"}):
    print(s)

## Studio

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

让我们在 Studio UI 中加载上面的图，它使用 `module-4/studio/map_reduce.py`，配置在 `module-4/studio/langgraph.json` 中。